# 第 08 课 · 可控圆柱与代理模型训练数据生成

> **学习模式：**独立编写 JAX CFD 求解器 | VS Code / Jupyter | 从公式到可运行代码

**本课目标**：实现一个受控旋转圆柱的 JAX CFD 时间推进函数，并生成 (状态, 动作, 下一状态) 数据集。

**前置知识**：第 03–07 课。旋转控制为教学示例，之后可扩展为壁面吹吸。

**操作约定**：先逐单元执行，再尝试“实验与练习”；若改变了函数定义或网格设置，请重启内核并执行全部单元格。JAX 第一次调用包含编译时间。

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from jax import lax
jax.config.update("jax_enable_x64", True)
print("JAX:", jax.__version__, "devices:", jax.devices())

## 1. 从 CFD 求解器到控制环境

你的毕设需要的关键数据结构是：

$$s_{t+1}=\mathcal F_{\rm CFD}(s_t,a_t),\qquad
\mathcal D=\{(s_t,a_t,s_{t+1})\}_{t=1}^N.$$

本课将动作 $a_t$ 定义为圆柱的无量纲角速度比（示例用角速度 $\omega$）：圆柱内部目标速度是刚体转动速度

$$u_s=-\omega(y-y_c),\qquad v_s=\omega(x-x_c).$$

这不是表面吹吸控制，而是一个**可以独立验证的入门控制边界**；后续吹吸控制还需要明确喷口位置、净质量流量与流固边界一致性。

我们仍使用周期域、光滑体积惩罚近似，数据适合训练流程原型，不宜未经更严格验证就用于报告真实减阻率。

In [ ]:
def grid(nx=64,ny=32,Lx=8.,Ly=4.):
 dx,dy=Lx/nx,Ly/ny
 x=jnp.arange(nx)*dx; y=jnp.arange(ny)*dy
 X,Y=jnp.meshgrid(x,y,indexing='ij')
 return dx,dy,X,Y

In [ ]:
def div(u,v,dx,dy):
 return (u-jnp.roll(u,1,0))/dx+(v-jnp.roll(v,1,1))/dy

In [ ]:
def grad(p,dx,dy):
 return (jnp.roll(p,-1,0)-p)/dx,(jnp.roll(p,-1,1)-p)/dy

In [ ]:
def lap(q,dx,dy):
 return (jnp.roll(q,-1,0)+jnp.roll(q,1,0)-2*q)/(dx*dx)+(jnp.roll(q,-1,1)+jnp.roll(q,1,1)-2*q)/(dy*dy)

In [ ]:
def symbol(nx,ny,dx,dy):
 kx=2*jnp.pi*jnp.fft.fftfreq(nx,d=dx)[:,None]
 ky=2*jnp.pi*jnp.fft.fftfreq(ny,d=dy)[None,:]
 s=-4*jnp.sin(kx*dx/2)**2/dx**2-4*jnp.sin(ky*dy/2)**2/dy**2
 return jnp.where(s==0,1.,s)

In [ ]:
def solve_poisson(rhs,s):
 rhs_hat=jnp.fft.fftn(rhs)
 p=jnp.fft.ifftn(rhs_hat/s).real
 return p-p.mean()

In [ ]:
def project(u,v,dx,dy,s):
 p=solve_poisson(div(u,v,dx,dy),s)
 px,py=grad(p,dx,dy)
 return u-px,v-py,p

In [ ]:
def advection(u,v,dx,dy):
 v_u=0.25*(v+jnp.roll(v,-1,0)+jnp.roll(v,1,1)+jnp.roll(jnp.roll(v,-1,0),1,1))
 u_v=0.25*(u+jnp.roll(u,1,0)+jnp.roll(u,-1,1)+jnp.roll(jnp.roll(u,1,0),-1,1))
 dux=(jnp.roll(u,-1,0)-jnp.roll(u,1,0))/(2*dx)
 duy=(jnp.roll(u,-1,1)-jnp.roll(u,1,1))/(2*dy)
 dvx=(jnp.roll(v,-1,0)-jnp.roll(v,1,0))/(2*dx)
 dvy=(jnp.roll(v,-1,1)-jnp.roll(v,1,1))/(2*dy)
 return u*dux+v_u*duy,u_v*dvx+v*dvy

In [ ]:
def geometry(nx,ny,dx,dy,cx=2,cy=2,r=.5,eps=None):
 eps=0.8*min(dx,dy) if eps is None else eps
 xu=(jnp.arange(nx)[:,None]+.5)*dx; yu=jnp.arange(ny)[None,:]*dy
 xv=jnp.arange(nx)[:,None]*dx; yv=(jnp.arange(ny)[None,:]+.5)*dy
 # simple nonperiodic distance OK when body away from edges
 ru=jnp.sqrt((xu-cx)**2+(yu-cy)**2)
 rv=jnp.sqrt((xv-cx)**2+(yv-cy)**2)
 masku=.5*(1-jnp.tanh((ru-r)/eps))
 maskv=.5*(1-jnp.tanh((rv-r)/eps))
 return masku,maskv

In [ ]:
def controlled_step(u,v,omega,dx,dy,dt,nu,S,mu,mv,eta,cx,cy,Uref=1.,gain=.5):
    nx,ny=u.shape
    # target solid motion evaluated at MAC faces
    y_u=jnp.arange(ny)[None,:]*dy
    x_v=jnp.arange(nx)[:,None]*dx
    us=-omega*(y_u-cy)
    vs= omega*(x_v-cx)
    def rhs(a,b):
        advu,advv=advection(a,b,dx,dy)
        drive=gain*(Uref-jnp.mean(a))
        return (-advu+nu*lap(a,dx,dy)+drive,
                -advv+nu*lap(b,dx,dy))
    def penal_project(a,b):
        a=(a+(dt/eta)*mu*us)/(1.+(dt/eta)*mu)
        b=(b+(dt/eta)*mv*vs)/(1.+(dt/eta)*mv)
        aa,bb,_=project(a,b,dx,dy,S)
        return aa,bb
    au,av=rhs(u,v)
    u1,v1=penal_project(u+dt*au,v+dt*av)
    au1,av1=rhs(u1,v1)
    u2,v2=penal_project(.5*(u+u1+dt*au1),.5*(v+v1+dt*av1))
    return u2,v2

## 2. `lax.scan` 进行动作序列滚动

每个动作持续固定数量 `substeps` 的 CFD 时间步。只保存每次动作结束时的状态，可以显著减少数据量。数据的张量形状：

- `states`: `(N+1, nx, ny, 2)`，最后维度表示 MAC 面上的两个速度数组（注意：它们是不同位置的量，直接堆叠仅为存储方便）。
- `actions`: `(N,)`，角速度控制输入。
- `next_states`: `(N, nx, ny, 2)`。

为了体现真正的 Markov 状态，需要同时保存完整状态而非仅凭某个单点探针。

In [ ]:
nx,ny,Lx,Ly=64,32,8.,4.
dx,dy,X,Y=grid(nx,ny,Lx,Ly)
S=symbol(nx,ny,dx,dy)
cx,cy,R=2.,2.,.5
mu,mv=geometry(nx,ny,dx,dy,cx,cy,R)
Re,Uref=80.,1.
nu=Uref*(2*R)/Re;dt=.018;eta=.035
u0=jnp.ones((nx,ny))*Uref
v0=jnp.zeros((nx,ny))
u0,v0,_=project(u0,v0,dx,dy,S)
actions=jnp.array([0.,0.25,0.5,0.0,-0.25,-0.5,0.0,0.2,-0.2,0.0])
substeps=30
@jax.jit
def rollout(u,v,actions):
 def one_action(state,omega):
  def one_step(_,uv):
   uu,vv=uv
   return controlled_step(uu,vv,omega,dx,dy,dt,nu,S,mu,mv,eta,cx,cy,Uref)
  nxt=lax.fori_loop(0,substeps,one_step,state)
  return nxt,nxt
 _,states_after=lax.scan(one_action,(u,v),actions)
 return states_after

u_record,v_record=rollout(u0,v0,actions)
states=np.stack([np.asarray(u_record),np.asarray(v_record)],axis=-1)
initial=np.stack([np.asarray(u0),np.asarray(v0)],axis=-1)
full_states=np.concatenate([initial[None,...],states],axis=0)
print('states:',full_states.shape,'actions:',actions.shape)
assert full_states.shape==(len(actions)+1,nx,ny,2)
assert np.isfinite(full_states).all()

## 3. 保存训练数据与基本完整性检查

`.npz` 压缩文件可直接被 NumPy、PyTorch 或 JAX 读取。除了状态/动作，还要保存物理参数、网格、边界条件、时间步长；否则数据难以复现。

注意这里的 `states` 是 MAC 交错面速度的打包表示，建立图像代理模型前，应考虑插值到统一的单元中心或使用明确的 staggered 数据通道。

In [ ]:
from pathlib import Path
outdir=Path('outputs');outdir.mkdir(exist_ok=True)
output_path=outdir/'cylinder_rotation_rollout.npz'
np.savez_compressed(output_path,
    states=full_states[:-1],
    actions=np.asarray(actions),
    next_states=full_states[1:],
    dt=dt, substeps=substeps, dx=dx, dy=dy,
    Re=Re, Uref=Uref, cx=cx, cy=cy, R=R,
    boundary_type='periodic_xy',
    cylinder_model='smoothed_volume_penalization')
loaded=np.load(output_path)
print('Saved:',output_path,'arrays:',loaded.files)
assert np.allclose(loaded['next_states'][0],loaded['states'][1])
assert loaded['actions'].shape[0]==loaded['states'].shape[0]

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(12,3.1),constrained_layout=True)
for j,idx in enumerate([0,4,9]):
 arr=full_states[idx,...,0]
 im=axs[j].imshow(arr.T,origin='lower',extent=(0,Lx,0,Ly),cmap='viridis',aspect='auto')
 axs[j].add_patch(plt.Circle((cx,cy),R,facecolor='white',edgecolor='black'))
 axs[j].set_title(f'State {idx}, t={idx*substeps*dt:.2f}')
 fig.colorbar(im,ax=axs[j],shrink=.8)
plt.show()

## 4. 为毕设做准备：从 CFD 环境到代理模型

1. **物理验证**：与独立求解器对照，做网格/时间步收敛。当前周期域不能代替标准入口/出口圆柱问题。
2. **数据设计**：覆盖动作幅值和频率；划分训练/验证时按**整个 rollout** 分组，避免相邻时间步数据泄漏。
3. **代理模型**：预测 $s_{t+1}$ 时，先检查短期速度误差，再检查多步误差、散度、力和奖励一致性。
4. **控制目标**：改为吹吸控制前，需要基于物理定义实现喷口边界、净流量约束和准确升阻力测量。

### 练习

- 将动作换成正弦序列 $\omega_k=A\sin(2\pi fk\Delta T)$，研究不同控制频率。
- 把每次动作的步数从 30 改为 60，讨论如何保证数值稳定。
- 将保存的数据读取回来，用 JAX 训练一个最简单的 `next_state ≈ state + f(state,action)` 模型（之后的专题课）。

**本阶段完成标准**：你能够独立写出二维不可压缩周期流动求解器；实现圆柱障碍与可控运动；跑出尾流；导出状态转移数据，并明确其科研精度边界。